# 0825_lsw_005_drift_label_robust

003/004에서 확인한 두 가지 drift(분포 drift, 라벨 판정기준 변화)를 바탕으로, "feature selection은 보류"
결정(scope_and_roadmap.md, dongjin 006-008/020 근거) 이후 실제로 시도할 만한 두 가지 방향을 검증한다.

1. **Recency weighting**: 최근 데이터에 더 큰 sample_weight를 줘서, 판정 기준이 최근으로 갈수록
   엄격해지는(0→1) 추세를 모델이 더 잘 반영하게 한다. (한 번에 하나만 바꾸는 원칙: 리샘플링/클래스가중치는
   섞지 않는다.)
2. **라벨 보정(최근 라벨 신뢰)**: 004의 라벨 클렌징(모순 라벨 "삭제")은 전체 기준 순손해(ΔTN -496, ΔFN
   +14)였고, 특히 type2에서 나빴다. 그런데 notes.md에서 확인했듯 모순 라벨의 81.4%가 "먼저 0 → 나중에
   1" 패턴이라 노이즈가 아니라 진짜 기준 변화일 가능성이 크다. 그렇다면 행을 지우기보다 **오래된 라벨을
   최근 라벨로 고쳐서(관측치는 유지)** 학습하는 게 더 나을 수 있다 — 이걸 검증한다.

데이터 전처리·평가 함수·검사유형별 5분리 구조는 003/004와 동일하게 재사용한다(비교 가능성 유지).


## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_lsw_005_drift_label_robust"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)


experiment: 0825_lsw_005_drift_label_robust


## 2. 데이터 로딩·전처리 (003/004와 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]

timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
train_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))]
valid_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time

print("rows_after_dedup:", len(clean_df))
pd.DataFrame(
    [
        {"split": name, "rows": int(mask.sum())}
        for name, mask in [("train", train_mask), ("validation", valid_mask), ("test", test_mask)]
    ]
).set_index("split")


rows_after_dedup: 391992


,rows
split,
train,235222
validation,78374
test,78396


## 3. 평가 함수 (003/004와 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def build_model(scale_pos_weight=1.0):
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    )


## 4. 검사유형별 subset 및 baseline (003/004 재현)

In [4]:
type_splits = {}
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]
    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)
    type_splits[inspection_type] = {
        "train": type_train_df,
        "valid": type_valid_df,
        "test": type_test_df,
        "feature_columns": type_feature_columns,
    }

baseline_results = {}
for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]

    model = build_model()
    model.fit(train_df[feature_columns], train_df[TARGET])

    valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
    threshold = select_threshold(valid_df[TARGET], valid_proba)
    test_proba = model.predict_proba(test_df[feature_columns])[:, 1]

    baseline_results[inspection_type] = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)

baseline_df = pd.DataFrame(baseline_results).T
baseline_df.index.name = "inspection_type"
baseline_df[["threshold", "tn", "fp", "fn", "tp", "pr_auc", "slip_rate", "volume_reduction"]]


,threshold,tn,fp,fn,tp,pr_auc,slip_rate,volume_reduction
inspection_type,,,,,,,,
0,0.000006,8857.0,7794.0,22.0,111.0,0.067265,0.165414,0.531920
1,0.000025,1869.0,8459.0,8.0,776.0,0.403144,0.010204,0.180964
2,0.000006,3492.0,14958.0,12.0,691.0,0.356827,0.017070,0.189268
3,0.000007,9700.0,20288.0,43.0,560.0,0.269568,0.071310,0.323463
4,0.000001,0.0,730.0,0.0,26.0,0.023284,0.000000,0.000000


## 5. Recency weighting 실험

003/004에서 확인한 사실: train 0.72% → val 0.45% → test 2.64% 순으로 불량 비율이 이동하고,
notes.md의 모순 라벨 분석에서 판정 기준이 시간이 갈수록 엄격해지는(0→1) 방향성이 뚜렷했다. 그렇다면
Train 내에서도 **최근 데이터일수록 "현재" 기준에 더 가까울 것**이므로, 오래된 데이터의 영향력을
줄이면 Validation/Test 시점의 기준을 더 잘 맞출 수 있다는 가설을 세운다.

`weight = 0.5 ** (age_days / half_life)` 형태의 지수감쇠 가중치를 Train에만 적용한다(각 검사유형의
Train 내 최신 timestamp를 기준점으로 사용 — Test 시점 정보를 쓰지 않으므로 누수 없음). 리샘플링이나
class_weight는 섞지 않는다(한 번에 하나만 바꾸는 원칙).

half-life 후보 3개(15일/30일/60일)를 검사유형별로 비교한다.


In [5]:
def recency_weights(timestamps, half_life_days):
    reference = timestamps.max()
    age_days = (reference - timestamps).dt.total_seconds() / 86400.0
    return 0.5 ** (age_days / half_life_days)


half_life_candidates = [15, 30, 60]
recency_results = []

for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]

    for half_life in half_life_candidates:
        weights = recency_weights(train_df[TIME_COLUMN], half_life)

        model = build_model()
        model.fit(train_df[feature_columns], train_df[TARGET], sample_weight=weights)

        valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
        threshold = select_threshold(valid_df[TARGET], valid_proba)
        test_proba = model.predict_proba(test_df[feature_columns])[:, 1]

        result = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)
        result["inspection_type"] = inspection_type
        result["half_life_days"] = half_life
        recency_results.append(result)

recency_results_df = pd.DataFrame(recency_results)
recency_results_df[["inspection_type", "half_life_days", "threshold", "tn", "fp", "fn", "tp", "pr_auc", "slip_rate", "volume_reduction"]]


,inspection_type,half_life_days,threshold,tn,fp,fn,tp,pr_auc,slip_rate,volume_reduction
0,0,15,9.371157e-06,2232,14419,17,116,0.059518,0.127820,0.134046
1,0,30,4.323885e-05,9632,7019,36,97,0.052864,0.270677,0.578464
2,0,60,2.169405e-06,3295,13356,14,119,0.056706,0.105263,0.197886
3,1,15,2.191449e-04,1902,8426,16,768,0.463696,0.020408,0.184160
4,1,30,8.197300e-05,2390,7938,9,775,0.361249,0.011480,0.231410
5,1,60,5.857875e-05,2457,7871,12,772,0.374299,0.015306,0.237897
6,2,15,1.581505e-06,57,18393,0,703,0.583437,0.000000,0.003089
7,2,30,1.240078e-05,3485,14965,13,690,0.493715,0.018492,0.188889
8,2,60,9.226349e-07,167,18283,3,700,0.560322,0.004267,0.009051
9,3,15,9.223004e-06,1828,28160,8,595,0.315331,0.013267,0.060958


## 6. Recency weighting 결과 비교 (baseline 대비 ΔTN/ΔFN)

In [6]:
recency_comparison_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    base = baseline_df.loc[inspection_type]
    for half_life in half_life_candidates:
        row = recency_results_df[
            (recency_results_df["inspection_type"] == inspection_type)
            & (recency_results_df["half_life_days"] == half_life)
        ].iloc[0]
        recency_comparison_rows.append(
            {
                "inspection_type": inspection_type,
                "half_life_days": half_life,
                "baseline_TN": int(base["tn"]), "baseline_FN": int(base["fn"]),
                "TN": row["tn"], "FN": row["fn"],
                "ΔTN": row["tn"] - int(base["tn"]),
                "ΔFN": row["fn"] - int(base["fn"]),
                "slip_rate": row["slip_rate"], "volume_reduction": row["volume_reduction"],
                "총비용(1:10)": row["total_cost_1:10"], "총비용(1:100)": row["total_cost_1:100"],
                "baseline_총비용(1:10)": base["total_cost_1:10"], "baseline_총비용(1:100)": base["total_cost_1:100"],
            }
        )

recency_comparison_df = pd.DataFrame(recency_comparison_rows).set_index(["inspection_type", "half_life_days"])
recency_comparison_df


baseline_TN  baseline_FN      TN    FN  \
inspection_type half_life_days                                           
0               15                     8857           22  2232.0  17.0   
                30                     8857           22  9632.0  36.0   
                60                     8857           22  3295.0  14.0   
1               15                     1869            8  1902.0  16.0   
                30                     1869            8  2390.0   9.0   
                60                     1869            8  2457.0  12.0   
2               15                     3492           12    57.0   0.0   
                30                     3492           12  3485.0  13.0   
                60                     3492           12   167.0   3.0   
3               15                     9700           43  1828.0   8.0   
                30                     9700           43  4680.0  22.0   
                60                     9700           43   628.0   5.0   
4               15                        0            0     2.0   1.0   
                30                        0            0     0.0   1.0   
                60                        0            0     0.0   0.0   

                                   ΔTN   ΔFN  slip_rate  volume_reduction  \
inspection_type half_life_days                                              
0               15             -6625.0  -5.0   0.127820          0.134046   
                30               775.0  14.0   0.270677          0.578464   
                60             -5562.0  -8.0   0.105263          0.197886   
1               15                33.0   8.0   0.020408          0.184160   
                30               521.0   1.0   0.011480          0.231410   
                60               588.0   4.0   0.015306          0.237897   
2               15             -3435.0 -12.0   0.000000          0.003089   
                30                -7.0   1.0   0.018492          0.188889   
                60             -3325.0  -9.0   0.004267          0.009051   
3               15             -7872.0 -35.0   0.013267          0.060958   
                30             -5020.0 -21.0   0.036484          0.156062   
                60             -9072.0 -38.0   0.008292          0.020942   
4               15                 2.0   1.0   0.038462          0.002740   
                30                 0.0   1.0   0.038462          0.000000   
                60                 0.0   0.0   0.000000          0.000000   

                                총비용(1:10)  총비용(1:100)  baseline_총비용(1:10)  \
inspection_type half_life_days                                              
0               15                14589.0     16119.0              8014.0   
                30                 7379.0     10619.0              8014.0   
                60                13496.0     14756.0              8014.0   
1               15                 8586.0     10026.0              8539.0   
                30                 8028.0      8838.0              8539.0   
                60                 7991.0      9071.0              8539.0   
2               15                18393.0     18393.0             15078.0   
                30                15095.0     16265.0             15078.0   
                60                18313.0     18583.0             15078.0   
3               15                28240.0     28960.0             20718.0   
                30                25528.0     27508.0             20718.0   
                60                29410.0     29860.0             20718.0   
4               15                  738.0       828.0               730.0   
                30                  740.0       830.0               730.0   
                60                  730.0       730.0               730.0   

                                baseline_총비용(1:100)  
inspection_type half_life_days                       
0               15                           

## 7. 결론 (recency weighting)

유형별로 half-life 3개 중 baseline을 이기는 게 있는지, 두 비용 시나리오(1:10/1:100) 모두에서
이기는지 확인한다.

| type | 최적 half-life | baseline 총비용(1:10→1:100) | 최적 총비용(1:10→1:100) | 판정 |
|---|---|---|---|---|
| 0 | 30일 | 8,014 → 9,994 | 7,379 → 10,619 | **1:10에서만 이김**(-635), 1:100은 오히려 손해(+625) — 비용비율에 따라 결론이 갈림 |
| 1 | 60일 | 8,539 → 9,259 | 7,991 → 9,071 | **두 시나리오 모두 이김**(-548 / -188) — 채택 |
| 2 | (없음) | 15,078 → 16,158 | 15,095 → 16,265 (30일 기준 최선) | 모든 half-life가 baseline보다 나쁘거나 무의미 — 기각 |
| 3 | (없음) | 20,718 → 24,588 | 25,528 → 27,508 (30일 기준 최선) | 모든 half-life가 baseline보다 나쁨 — 기각 |
| 4 | (없음) | 730 → 730 | 730 → 730 (60일) | 표본 26건, 사실상 변화 없음 — 판단 보류 |

**해석**:

- **type1은 명확한 개선**이다. half-life=60일에서 ΔTN=+588, ΔFN=+4 — TN 이득이 FN 손해를 압도해서
  두 비용비율 모두에서 총비용이 낮아진다. 003/004에서 이미 표본이 충분했던(train 양성 다수) 유형이라
  신뢰할 만하다.
- **type0은 비용비율에 따라 결론이 갈린다.** half-life=30일이 1:10에서는 이기지만(FP를 크게 줄이는
  대신 FN이 22→36으로 늘어나는 교환) 1:100에서는 진다(FN 증가분이 100배 비용으로 커지기 때문). 안전
  최우선(1:100에 가까운 정책)이면 채택하지 않는 게 맞다.
- **type2/3은 recency weighting이 전반적으로 해롭다** — half-life가 짧을수록(특히 15일) 임계값이
  극단적으로 낮아지면서 TN이 붕괴한다(type2 hl=15: TN 3492→57). 오래된 데이터를 급격히 죽이면 오히려
  학습 표본이 사실상 최근 며칠로 좁아져 과적합하는 것으로 보인다.
- **type4는 표본(양성 26건)이 너무 적어 half-life 선택이 우연에 가깝다** — hl=60이 baseline과 완전히
  동일한 결과라 사실상 아무 정보도 얻지 못했다.

**요약**: recency weighting은 만능이 아니라 **type1에서만 두 비용비율 모두 확실한 개선**이고,
type0은 비용정책에 따라 갈리며, type2/3/4는 채택하지 않는다. "최근 데이터에 기준이 더 엄격해진다"는
drift 가설 자체는 003/004의 시간별 불량률 이동과 방향이 같지만, 그 가설을 곧바로 sample_weight로
치환하는 것이 유형마다 통했다 안 통했다 하므로 **유형별로 검증 없이 일괄 적용해서는 안 된다.**


## 8. 라벨 판정기준 변화 재검토 — "삭제" 대신 "최근 라벨로 통일"

004의 라벨 클렌징(모순 라벨 행 삭제)은 전체 기준 순손해(ΔTN -496, ΔFN +14)였고, type2에서 TN
3,072건을 잃으면서 FN은 10건만 줄어 나쁜 거래였다. notes.md의 분석(전체 데이터에서 모순 그룹 70개
중 57개(81.4%)가 "먼저 0 → 나중에 1")에 따르면 이 모순은 노이즈가 아니라 **판정 기준이 실제로
엄격해진 결과**일 가능성이 크다. 그렇다면 오래된(느슨한 기준 시절) 라벨을 지우기보다 **그룹 내에서
가장 최근 timestamp의 라벨로 통일**하는 게 더 나을 수 있다 — 행을 잃지 않으면서 "현재 기준"으로
정답을 고쳐주는 접근이다.

Train 안에서만 수행한다(Validation/Test는 원본 라벨 그대로 평가 — 실제 운영에서도 미래 라벨을 알 수
없으므로).


In [7]:
def label_correct_recent(train_df, feature_columns):
    df = train_df.copy()
    nunique_classes = df.groupby(feature_columns)[TARGET].transform("nunique")
    contradictory_mask = (nunique_classes > 1).to_numpy()

    n_relabeled = 0
    if contradictory_mask.any():
        group_id = df.groupby(feature_columns, sort=False).ngroup()
        contradictory_group_id = group_id.loc[contradictory_mask]
        contradictory_target = df.loc[contradictory_mask, TARGET]

        latest_time_idx = df.loc[contradictory_mask, TIME_COLUMN].groupby(contradictory_group_id).idxmax()
        group_to_label = pd.Series(
            df.loc[latest_time_idx.to_numpy(), TARGET].to_numpy(),
            index=latest_time_idx.index,
        )

        new_labels = contradictory_group_id.map(group_to_label)
        n_relabeled = int((new_labels.to_numpy() != contradictory_target.to_numpy()).sum())
        df.loc[contradictory_mask, TARGET] = new_labels.to_numpy()

    return df, n_relabeled


label_correction_results = []
for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]

    corrected_train_df, n_relabeled = label_correct_recent(train_df, feature_columns)

    model = build_model()
    model.fit(corrected_train_df[feature_columns], corrected_train_df[TARGET])

    valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
    threshold = select_threshold(valid_df[TARGET], valid_proba)
    test_proba = model.predict_proba(test_df[feature_columns])[:, 1]

    result = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)
    result["inspection_type"] = inspection_type
    result["n_relabeled"] = n_relabeled
    label_correction_results.append(result)

label_correction_df = pd.DataFrame(label_correction_results).set_index("inspection_type")
label_correction_df[["n_relabeled", "threshold", "tn", "fp", "fn", "tp", "pr_auc", "slip_rate", "volume_reduction"]]


,n_relabeled,threshold,tn,fp,fn,tp,pr_auc,slip_rate,volume_reduction
inspection_type,,,,,,,,,
0,12,7.904678e-07,2832,13819,12,121,0.079122,0.090226,0.170080
1,0,2.536666e-05,1869,8459,8,776,0.403144,0.010204,0.180964
2,14,5.199254e-06,1677,16773,8,695,0.257675,0.011380,0.090894
3,0,6.984155e-06,9700,20288,43,560,0.269568,0.071310,0.323463
4,0,1.142581e-06,0,730,0,26,0.023284,0.000000,0.000000


## 9. 라벨 보정 비교 — baseline vs 004 라벨 클렌징(삭제) vs 최근 라벨 보정

In [8]:
# 004의 라벨 클렌징(삭제) 결과를 이 노트북에서도 동일 로직으로 재현해 나란히 비교한다.
label_deletion_results = []
for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]

    nunique_classes = train_df.groupby(feature_columns)[TARGET].transform("nunique")
    contradictory_mask = nunique_classes > 1
    train_clean_df = train_df.loc[~contradictory_mask]

    model = build_model()
    model.fit(train_clean_df[feature_columns], train_clean_df[TARGET])

    valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
    threshold = select_threshold(valid_df[TARGET], valid_proba)
    test_proba = model.predict_proba(test_df[feature_columns])[:, 1]

    result = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)
    result["inspection_type"] = inspection_type
    result["n_removed"] = int(contradictory_mask.sum())
    label_deletion_results.append(result)

label_deletion_df = pd.DataFrame(label_deletion_results).set_index("inspection_type")

label_compare_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    base = baseline_df.loc[inspection_type]
    delete = label_deletion_df.loc[inspection_type]
    correct = label_correction_df.loc[inspection_type]
    label_compare_rows.append(
        {
            "inspection_type": inspection_type,
            "baseline_TN": int(base["tn"]), "baseline_FN": int(base["fn"]),
            "삭제_TN": int(delete["tn"]), "삭제_FN": int(delete["fn"]),
            "삭제_ΔTN": int(delete["tn"]) - int(base["tn"]), "삭제_ΔFN": int(delete["fn"]) - int(base["fn"]),
            "최근라벨보정_TN": int(correct["tn"]), "최근라벨보정_FN": int(correct["fn"]),
            "최근라벨보정_ΔTN": int(correct["tn"]) - int(base["tn"]), "최근라벨보정_ΔFN": int(correct["fn"]) - int(base["fn"]),
            "baseline_총비용(1:10)": base["total_cost_1:10"],
            "삭제_총비용(1:10)": delete["total_cost_1:10"],
            "최근라벨보정_총비용(1:10)": correct["total_cost_1:10"],
            "baseline_총비용(1:100)": base["total_cost_1:100"],
            "삭제_총비용(1:100)": delete["total_cost_1:100"],
            "최근라벨보정_총비용(1:100)": correct["total_cost_1:100"],
        }
    )

label_compare_df = pd.DataFrame(label_compare_rows).set_index("inspection_type")
label_compare_df


,baseline_TN,baseline_FN,삭제_TN,삭제_FN,삭제_ΔTN,삭제_ΔFN,최근라벨보정_TN,최근라벨보정_FN,최근라벨보정_ΔTN,최근라벨보정_ΔFN,baseline_총비용(1:10),삭제_총비용(1:10),최근라벨보정_총비용(1:10),baseline_총비용(1:100),삭제_총비용(1:100),최근라벨보정_총비용(1:100)
inspection_type,,,,,,,,,,,,,,,,
0,8857,22,11433,46,2576,24,2832,12,-6025,-10,8014.0,5678.0,13939.0,9994.0,9818.0,15019.0
1,1869,8,1869,8,0,0,1869,8,0,0,8539.0,8539.0,8539.0,9259.0,9259.0,9259.0
2,3492,12,420,2,-3072,-10,1677,8,-1815,-4,15078.0,18050.0,16853.0,16158.0,18230.0,17573.0
3,9700,43,9700,43,0,0,9700,43,0,0,20718.0,20718.0,20718.0,24588.0,24588.0,24588.0
4,0,0,0,0,0,0,0,0,0,0,730.0,730.0,730.0,730.0,730.0,730.0


## 10. 최종 결론 및 다음 단계

### 라벨 보정("최근 라벨로 통일") 결과

| type | n_relabeled | baseline 총비용(1:10→1:100) | 삭제(004) 총비용 | 최근라벨보정 총비용 | 판정 |
|---|---:|---|---|---|---|
| 0 | 12 | 8,014 → 9,994 | **5,678 → 9,818**(둘 다 이김) | 13,939 → 15,019(둘 다 큰 손해) | 삭제가 압도적으로 낫다. 보정은 채택 안 함 |
| 1 | 0 | — | — | — | 모순 그룹 없음, 해당 없음 |
| 2 | 14 | 15,078 → 16,158 | 18,050 → 18,230(둘 다 손해) | **16,853 → 17,573**(손해지만 삭제보다 덜 나쁨) | baseline이 여전히 최선. 보정은 삭제보다는 낫지만 baseline은 못 이긴다 |
| 3 | 0 | — | — | — | 모순 그룹 없음, 해당 없음 |
| 4 | 0 | — | — | — | 모순 그룹 없음, 해당 없음 |

**가설 검증 결과**: "행을 지우기보다 최근 라벨로 고쳐서 유지하는 게 낫다"는 **부분적으로만 맞았다.**

- type2에서는 방향이 맞았다 — 보정(ΔTN -1,815/ΔFN -4)이 삭제(ΔTN -3,072/ΔFN -10)보다 손해가 작다.
  같은 발상(오래된 라벨이 틀렸다고 보고 고친다)이라도 **행을 통째로 버리는 것보다 라벨만 고쳐 표본을
  유지하는 쪽이 덜 파괴적**이라는 근거가 됐다. 다만 그래도 baseline(아무것도 안 함)보다는 나쁘다 —
  즉 type2는 "이 모순 라벨을 건드리지 않는 것"이 최선이라는 004의 결론이 유지된다.
- type0에서는 오히려 정반대로 나왔다 — 보정이 baseline보다도 훨씬 나쁘다(총비용이 거의 2배). 원인은
  TN이 8,857→2,832로 붕괴한 것: 12건의 라벨을 0→1로 바꾸자 모델이 전반적으로 양성 쪽에 더 민감해져,
  Slip Rate ≤1% 제약을 맞추려는 임계값이 훨씬 낮아지고 FP가 급증했다. type0은 표본이 작아(양성
  Train 57건 수준) 소수 라벨 변경의 파급力이 과도하게 크게 작동한 것으로 보인다. **type0은 (004의
  결론대로) 모순 행을 "삭제"하는 쪽이 유일한 승자다 — 보정은 쓰지 않는다.**
- 결론적으로 004의 "라벨 클렌징(삭제) 일괄 적용은 순손해" 판단 자체는 바뀌지 않는다. 다만 **유형별로
  다르게 적용해야 한다**는 004의 결론에 세부 사항이 하나 더 붙는다: type0은 삭제, type2는 아무것도
  안 하는 것(보정도 삭제도 아님)이 최선이다.

### 검사유형별 최적 조합 갱신 (003+004+005 종합)

004의 12절 최종 종합표(`0824_lsw_004`, 검사유형별 최적 옵션의 정확한 총비용)와 이번 노트북의
recency weighting/라벨보정 결과를 직접 비교했다.

| type | 최적 기법(1:10) | 비용 | 최적 기법(1:100) | 비용 | 비고 |
|---|---|---:|---|---:|---|
| 0 | label_cleansing(삭제) | 5,678 | label_cleansing(삭제) | 9,818 | recency weighting(hl=30)은 7,379/10,619로 더 나쁨 — 순위 변화 없음 |
| 1 | undersample | 6,209 | undersample | 7,289 | **recency weighting(hl=60)은 7,991/9,071로 undersample보다 나쁨** — 처음엔 새 후보로 착각했으나 004 원자료와 직접 비교하니 undersample이 여전히 이긴다. 순위 변화 없음 |
| 2 | undersample | 10,541 | undersample | 14,411 | 라벨 삭제/보정 둘 다 baseline(15,078/16,158)보다 나쁨을 재확인. drift로 보이는 모순 라벨은 건드리지 않는 게 맞다 |
| 3 | adasyn | 17,515 | smote | 22,456 | recency weighting 전부 baseline(20,718/24,588)보다 나쁨 — 기각 |
| 4 | undersample | 715 | undersample | 715 | 표본 부족(양성 26건)으로 recency weighting도 라벨 보정도 유의미한 신호 없음 |

**이번 노트북(005)의 실질적 기여**: 두 가지 새 기법(recency weighting, 라벨 최근값 보정) 모두
**검사유형별 최적 조합의 순위 자체를 바꾸지는 못했다.** 즉 004까지의 결론(유형별 조합표)이 이번
검증에서도 그대로 버틴다는 게 이 노트북의 핵심 성과다 — 뭔가 새로 채택된 게 아니라, "이미 나온
결론이 두 가지 새로운 시도에도 흔들리지 않는다"는 걸 확인한 것.

### 재학습 트리거는 이번 범위에서 제외

정적 데이터셋(132일 스냅샷)이라 "언제 재학습할지"를 실제 운영 트리거로 시뮬레이션할 근거가 약하다.
게다가 이번에 다룬 문제는 P(y|X) 자체가 바뀌는 concept drift라, PSI 같은 분포 기반 트리거로는 애초에
감지되지 않는다(X 분포는 안 변했으므로) — 성능/라벨 기반 모니터링이 있어야 하는데 이 데이터로는 검증
불가능하다. 따라서 재학습 트리거는 **구현하지 않고, 이 한계만 문서에 남긴다.**

### 다음 세션 TODO

1. 이상치 탐지 재구현(라벨 무관 극단값 기준) — 여전히 미착수.
2. Phase 4(비지도 이상탐지)로 이동 가능 — feature selection/drift/라벨강건 모두 결론이 남(어느 것도
   004까지의 검사유형별 최적 조합표를 못 이겼다는 게 최종 결론).
